# 04 · matplotlib：把结论画出来

**时间预算**：3 小时。

前三课都在"算"，今天开始"说"。图是最有力的表达工具，**也是最好用的撒谎工具**——
同一份数据，换个画法能得出相反的结论，而且不伪造任何数字。

### 这节课怎么用（和前三课不一样了）

从这节课开始改成**讲一段、练一段**：

- 每个知识点后面**立刻**跟一个 2–5 分钟的小练习，做完再往下
- 小练习都写了"你应该观察到什么"——**看到了才算过**
- 全部小练习做完，最后有 **4 道综合大练习**

> **为什么改成这样**：知识有保质期。全堆到最后做，前面讲的东西已经凉了。
> 而且小练习的作用是**当场暴露误解**——错在刚讲完的地方，改起来最便宜。

## 1. 先解决中文乱码（第 1 课埋的坑）

matplotlib 默认字体里没有汉字，中文会变成方框。**这两行要写在所有画图代码之前**。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from fontTools.diff import color

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False  # 不设的话负号会变成方框
plt.rcParams["figure.dpi"] = 110

print("matplotlib 版本:", __import__("matplotlib").__version__)
df = pd.read_csv("../data/raw/titanic.csv")
print("数据:", df.shape)

### 🔹 小练习 1 · 两分钟

把字体改成**一个不存在的名字**，画个中文标题。

**你应该观察到两件事**：

1. 标题里的中文变成了**方框**
2. **屏幕会刷出一大片提示**（几十行），里面有两类关键信息：
   - `findfont: ... not found` —— 这个字体没找到
   - `UserWarning: Glyph 26041 (...) missing from font(s)` —— 这个字的字形缺失

**别被这一屏吓到**，它只是提示，程序没崩。

> 这条经验很值钱：以后中文变成方框时，**先往上翻找这两类提示**——
> 它们会直接告诉你"字体没配上"，你不用猜。

In [ ]:
plt.rcParams["font.sans-serif"] = ["不存在的字体"]

fig, ax = plt.subplots(figsize=(4, 2))
ax.set_title("中文会变成方框吗")
plt.show()

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]  # 记得改回来
print("字体已改回:", plt.rcParams["font.sans-serif"])

## 2. fig 和 ax：matplotlib 的心智模型

这两个词的区分是分水岭，搞懂了后面都顺，搞不懂就一直抄代码。

| 概念 | 中文 | 是什么 |
| :--- | :--- | :--- |
| `Figure` | 画布 | 一整张图，可以理解成"那张纸" |
| `Axes` | 坐标系 | 画布上的一个绘图区，**你 90% 的操作都是对它做的** |

```python
fig, ax = plt.subplots()          # 一张画布 + 一个坐标系
fig, axes = plt.subplots(2, 2)    # 一张画布 + 四个坐标系
```

`plt.plot(...)` 那种一行写法，是"在最近用过的画布上画"的快捷方式——
单张图能用，一画子图就出怪问题。**从一开始就用 `fig, ax`，别养成偷懒的习惯。**

In [ ]:
# 两种写法对比
plt.plot([1, 2, 3], [1, 4, 9])  # 快捷写法
plt.title("快捷写法")
plt.show()

fig, ax = plt.subplots(figsize=(4, 3))  # 面向对象写法
ax.plot([1, 2, 3], [1, 4, 9])
ax.set_title("面向对象写法")
plt.show()

### 🔹 小练习 2 · 三分钟

用 `subplots(1, 2)` 在**同一张画布**上画两条曲线：左边 `y = x²`，右边 `y = x³`，各自带标题。

x 用 `np.linspace(-3, 3, 100)`（第 1 课学过的）。

**你应该观察到**：`plt.show()` 只需要写**一次**，不用在两段代码之间来回切。
这就是 `fig, ax` 相比快捷写法的直接好处。

In [ ]:
# 你的代码
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
x = np.linspace(-3, 3, 100)
ax[0].plot(x, x**2)
ax[1].plot(x, x**3)
plt.show()
# 提示：fig, axes = plt.subplots(1, 2, figsize=(8, 3))
#       然后用 axes[0] 和 axes[1]

### 一张"合格"的图，至少要有四样东西

很多人画的图只有线和数据，别人看不懂。一张能给人看的图要回答四个问题：

1. **这张图在说什么？** → 标题
2. **横轴纵轴是什么？单位？** → 轴标签
3. **不同颜色代表什么？** → 图例
4. **我最想让你看哪一点？** → 注释（annotate）

第 4 条最容易被忽略，但它决定了读者是"3 秒抓到重点"还是"自己编一个结论"。

In [ ]:
survival = df.groupby("Pclass")["Survived"].mean()
counts = df.groupby("Pclass")["Survived"].count()

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    ["1 等舱", "2 等舱", "3 等舱"],
    survival.values,
    color=["#4C72B0", "#DD8452", "#55A868"],
)

ax.set_title("舱位越高，生还概率越大", fontsize=14, pad=12)
ax.set_xlabel("客舱等级")
ax.set_ylabel("生还率")
ax.set_ylim(0, 0.75)  # 从 0 开始，不夸大差异
ax.grid(axis="y", alpha=0.3, linestyle="--")
ax.set_axisbelow(True)

for bar, value, n in zip(bars, survival.values, counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.02,
        f"{value:.0%}\n({n} 人)",
        ha="center",
        fontsize=10,
    )

ax.annotate(
    "3 等舱只有 1 等舱的 38%",
    xy=(2, survival.values[2]),
    xytext=(1.2, 0.58),
    arrowprops=dict(arrowstyle="->", color="gray"),
    fontsize=10,
    color="dimgray",
)

fig.tight_layout()
plt.show()

### 🔹 小练习 3 · 三分钟 · 换个角度理解"什么让图能读"

把上面那张图**做三处改动**，每改一处跑一次：

1. 删掉标题
2. 删掉 `ax.text(...)` 那个 for 循环（不给柱子标数值）
3. 把 `ax.set_ylim(0, 0.75)` 改成 `ax.set_ylim(0.2, 0.7)`

**你应该观察到**：三处改动里，**第 3 处对"结论"的影响最大**——
它让三等舱看起来几乎没有生还希望。而这一处只是改了一个数字，没动任何数据。

> 这就是第 6 节要专门讲的东西。先埋个种子。

In [ ]:
# 你的代码：复制上面那一格，逐条改，每次跑完看区别
survival = df.groupby("Pclass")["Survived"].mean()
counts = df.groupby("Pclass")["Survived"].count()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    ["一等舱", "二等仓", "三等仓"],
    survival.values,
    color=["#4C72B0", "#DD8452", "#55A868"],
)
ax.set_title("仓位越高，生还概率越大")
ax.set_xlabel("客舱等级")
ax.set_ylabel("生还率")
ax.set_ylim(0, 0.75)
for bar, value, count in zip(bars, survival.values, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.02,
        f"{value:.0%}\n（{count}人）",
        ha="center",
        fontsize=10,
    )
ax.annotate(
    "3等仓只有1等仓的38%",
    xy=(2, survival.values[2]),
    xytext=(1.2, 0.58),
    arrowprops=dict(arrowstyle="->", color="gray"),
    fontsize=10,
    color="dimgray",
)

fig.tight_layout()
plt.show()

## 3. 五种基本图形：选对图是这门手艺的一半

**同一个问题用错图形，等于白画。**

| 图形 | 方法 | 回答什么问题 | 典型误用 |
| :--- | :--- | :--- | :--- |
| 折线图 | `plot` | **随时间**怎么变 | 拿来比类别大小 |
| 柱状图 | `bar` | 不同**类别**谁高谁低 | 柱数超过 10 根还在硬塞 |
| 直方图 | `hist` | 一个变量的**分布**长什么样 | 和柱状图混为一谈 |
| 散点图 | `scatter` | 两个变量**有关系吗** | 点太多糊成一片 |
| 箱线图 | `boxplot` | 多组的**分布**对比 | 说不清箱子的含义 |

先看折线图——唯一适合"时间"的图形。

In [ ]:
rng = np.random.default_rng(42)
months = [f"{m}月" for m in range(1, 13)]
orders = 100 + np.arange(12) * 8 + rng.normal(0, 12, 12)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(months, orders, marker="o", linewidth=2, color="#4C72B0")
ax.set_title("月度订单量（模拟数据）")
ax.set_ylabel("订单数")
ax.set_ylim(0, orders.max() * 1.2)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

### 🔹 小练习 4 · 两分钟

把上面这张图改成：**方块点**（`marker="s"`）、**红色**、**虚线**（`linestyle="--"`）。

然后**删掉那一行 `ax.set_ylim(...)`**，再跑一次。

**你应该观察到**：删掉 ylim 之后，y 轴不再从 0 开始了。
看一下这一版和原来那版的观感差别——**同样的数据，波动看起来"更剧烈"了**。

In [ ]:
# 你的代码
rng = np.random.default_rng(42)
months = [f"{m}月" for m in range(1, 13)]
orders = 100 + np.arange(12) * 8 + rng.normal(0, 12, 12)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(months, orders, marker="s", linewidth=2, color="#4C72B0", linestyle="--")
ax.set_title("月度订单量（模拟数据）")
ax.set_ylabel("订单数")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

### 直方图 vs 柱状图：最容易混的一对

- **柱状图**：横轴是**类别**（舱位、港口、性别），柱子之间有空隙，看的是"谁高"
- **直方图**：横轴是**数值区间**（0-10、10-20……），柱子挨着，看的是"形状"

**判断方法**：横轴的类别能不能换顺序？能换是柱状图，不能换（有大小顺序）就是直方图。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

axes[0].hist(df["Age"].dropna(), bins=20, color="#4C72B0", edgecolor="white")
axes[0].set_title("年龄分布（20 个区间）")
axes[0].set_xlabel("年龄")
axes[0].set_ylabel("人数")

axes[1].hist(df["Fare"], bins=40, color="#DD8452", edgecolor="white")
axes[1].set_title("票价分布（40 个区间）")
axes[1].set_xlabel("票价")

fig.tight_layout()
plt.show()
print("  ^ 年龄接近对称；票价严重右偏 —— 绝大多数人票价很低，少数人极高")

### 🔹 小练习 5 · 四分钟 · 直方图的核心

用**同一份票价数据**画三张直方图：`bins=5`、`bins=20`、`bins=60`。

**你应该观察到**：

- `bins=5`：太粗，看不到"右偏"，看起来像个普通的柱子
- `bins=60`：太细，出现一堆锯齿状的噪声
- `bins=20` 左右：形状最清楚

> **记住这条**：直方图的分箱数会改变你看到的"形状"。
> 别人给你看直方图时，**先问一句 bins 是多少**——换一个分箱数，故事可能就不一样了。

In [ ]:
# 你的代码
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
axes[0].hist(df["Fare"], bins=5, color="#4C72B0", edgecolor="white")
axes[1].hist(df["Fare"], bins=20, color="#4C72B0", edgecolor="white")
axes[2].hist(df["Fare"], bins=60, color="#4C72B0", edgecolor="white")

axes[0].set_title("5 个区间")
axes[1].set_title("20 个区间")
axes[2].set_title("60 个区间")

axes[0].set_xlabel("票价")
axes[1].set_xlabel("票价")
axes[2].set_xlabel("票价")

axes[0].set_ylabel("人数")
axes[1].set_ylabel("人数")
axes[2].set_ylabel("人数")

fig.tight_layout()
plt.show()

# 提示：fig, axes = plt.subplots(1, 3, figsize=(13, 3))

### 散点图：看两个变量有没有关系

`alpha` 是透明度。**点一多就必须调它**，否则会糊成一团黑。

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

for pclass, color in [(1, "#4C72B0"), (2, "#DD8452"), (3, "#55A868")]:
    subset = df[df["Pclass"] == pclass]
    ax.scatter(
        subset["Age"],
        subset["Fare"],
        alpha=0.5,
        s=25,
        label=f"{pclass} 等舱",
        color=color,
    )

ax.set_title("年龄与票价的关系")
ax.set_xlabel("年龄")
ax.set_ylabel("票价")
ax.legend(title="舱位")
fig.tight_layout()
plt.show()
print("  ^ 三个颜色明显分层：票价这条轴上，舱位分得比年龄清楚得多")

### 🔹 小练习 6 · 两分钟

把 `alpha` 从 `0.5` 改成 `1.0`，跑一次；再改成 `0.15`，跑一次。

**你应该观察到**：

- `alpha=1.0`：三个颜色忠实呈现，但**重叠的地方被后面的点盖住了**，看不出哪块密
- `alpha=0.15`：颜色混在一起，但**密集的区域自己浮出来了**

> 这就是散点图的意义：**不是看每个点，是看点的密度分布。**

In [ ]:
# 你的代码
fig, ax = plt.subplots(figsize=(6, 4))

for pclass, color in [(1, "#4C72B0"), (2, "#DD8452"), (3, "#55A868")]:
    subset = df[df["Pclass"] == pclass]
    ax.scatter(
        subset["Age"],
        subset["Fare"],
        alpha=0.15,
        s=25,
        label=f"{pclass} 等舱",
        color=color,
    )

ax.set_title("年龄与票价的关系")
ax.set_xlabel("年龄")
ax.set_ylabel("票价")
ax.legend(title="舱位")

fig.tight_layout()
plt.show()

### 箱线图：怎么看懂那个"箱子"

信息密度极高，但**很多人画了箱线图却说不清箱子的含义**：

```
        ┬      ← 上须：不是极值的最大值
    ┌───┴───┐
    │       │  ← 箱体：中间 50% 的人（25% 到 75% 分位）
    ├───────┤  ← 箱内横线：中位数
    │       │
    └───┬───┘
        ┴      ← 下须
        °      ← 单独的点：离群值
```

**注意参数名**：matplotlib 3.9 之后 `labels=` 改成了 `tick_labels=`。
网上大量教程还是旧写法，抄下来会直接报 `TypeError`。

In [ ]:
groups = [df[df["Pclass"] == p]["Fare"].values for p in [1, 2, 3]]

fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot(
    groups,
    tick_labels=["1 等舱", "2 等舱", "3 等舱"],
    patch_artist=True,
    boxprops=dict(facecolor="#B8D4E8"),
)
ax.set_title("各舱位票价分布")
ax.set_ylabel("票价")
ax.set_ylim(0, 300)  # 上面还有 512 的极端值，先截掉让主图看得清
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

### 🔹 小练习 7 · 两分钟 · 这题解释了箱线图的价值

把 `ax.set_ylim(0, 300)` 这一行**注释掉**，再跑一次。

**你应该观察到**：因为有个乘客的票价是 **512**，整张图被压扁了，
三个箱体挤在很下面，**反而看不清差别了**。

现在把 ylim 加回来，对比一下。

> **这个对比就是箱线图的价值所在**：箱体只由 25%/50%/75% 三个分位数决定，
> **不受极端值影响**——那个 512 不会把中位数拉走。而如果你改用"每组平均票价"的柱状图，
> 1 等舱的柱子会被这 512 直接抬高。**这就是为什么比分布要用箱线图，不用平均值柱状图。**

In [ ]:
# 你的代码
groups = [df[df["Pclass"] == p]["Fare"].values for p in [1, 2, 3]]
fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot(groups, tick_labels=["1 等舱", "2 等舱", "3 等舱"])

ax.set_title("各舱位票价分布")
ax.set_ylabel("票价")
ax.grid(axis="y", alpha=0.3)
# ax.set_ylim(0, 300)
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
plt.show()

### 热力图：看多个变量之间的关系

`df.corr()` 算出的相关系数是一张表，用热力图才看得出哪几对关系强。

**配色有个原则**：相关系数的范围是 −1 到 +1，**中间是 0**，
所以必须用**发散色系**（`RdBu_r`、`coolwarm` 这种），让 0 落在中间色、正负分居两端。

In [ ]:
numeric = df[["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"]]
corr = numeric.corr()

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
fig.colorbar(im, ax=ax, label="相关系数")

ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.columns)

for i in range(len(corr)):  # 把数值也写进格子
    for j in range(len(corr)):
        ax.text(
            j,
            i,
            f"{corr.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=8,
            color="white" if abs(corr.iloc[i, j]) > 0.5 else "black",
        )

ax.set_title("数值列相关性矩阵")
fig.tight_layout()
plt.show()
print("  ^ 对角线永远是 1；SibSp 和 Parch 是 0.41，说明大家庭往往两者都多")

### 🔹 小练习 8 · 两分钟 · 配色不是审美问题

把 `cmap="RdBu_r"` 改成 `cmap="viridis"`，跑一次。

**你应该观察到**：viridis 是一套**从暗到亮**的顺序色系，它**没有"中间"**。
结果就是——你没法一眼看出哪对变量是**负相关**，因为深色可能代表 −0.55，
也可能代表 +0.1，**颜色本身不再告诉你方向**。

> **结论**：涉及正负的数据（相关系数、增长率、盈亏）一律用发散色系。
> 只涉及大小、没有正负的数据（人数、金额）才用顺序色系。

In [ ]:
# 你的代码numeric = df[["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"]]
corr = numeric.corr()

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(corr.values, cmap="viridis")
fig.colorbar(im, ax=ax, label="相关系数")

ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.columns)

for i in range(len(corr)):  # 把数值也写进格子
    for j in range(len(corr)):
        ax.text(
            j,
            i,
            f"{corr.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=8,
            color="white" if abs(corr.iloc[i, j]) > 0.5 else "black",
        )

ax.set_title("数值列相关性矩阵")
fig.tight_layout()
plt.show()
print("  ^ 对角线永远是 1；SibSp 和 Parch 是 0.41，说明大家庭往往两者都多")

## 4. 子图布局

`plt.subplots(行, 列)` 返回的 `axes` 是**二维数组**，用 `axes[行, 列]` 取。
只有一行或一列时会退化成一维，所以 `axes[0]`、`axes[1]` 是最常用的写法。

`constrained_layout=True` 会帮你自动排布，省掉手动调间距的痛苦。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)

sex_survival = df.groupby("Sex")["Survived"].mean()
axes[0, 0].bar(["女性", "男性"], sex_survival.values, color=["#DD8452", "#4C72B0"])
axes[0, 0].set_title("性别 vs 生还率")
axes[0, 0].set_ylabel("生还率")
axes[0, 0].set_ylim(0, 1)
for i, value in enumerate(sex_survival.values):
    axes[0, 0].text(i, value + 0.03, f"{value:.0%}", ha="center")

axes[0, 1].hist(df["Fare"], bins=40, color="#55A868", edgecolor="white")
axes[0, 1].set_title("票价分布")
axes[0, 1].set_xlabel("票价")
axes[0, 1].set_ylabel("人数")

df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
family = df.groupby("FamilySize")["Survived"].agg(["count", "mean"])
family = family[family["count"] >= 5]  # 样本太少的组没有统计意义
axes[1, 0].plot(family.index, family["mean"], marker="o", color="#C44E52")
axes[1, 0].set_title("家庭规模 vs 生还率")
axes[1, 0].set_xlabel("家庭人数")
axes[1, 0].set_ylabel("生还率")
axes[1, 0].set_ylim(0, 0.9)
axes[1, 0].grid(alpha=0.3)

port = df.groupby("Embarked")["Survived"].mean().sort_values()
axes[1, 1].barh(port.index, port.values, color="#8172B3")
axes[1, 1].set_title("登船港 vs 生还率")
axes[1, 1].set_xlabel("生还率")
axes[1, 1].set_xlim(0, 0.7)

fig.suptitle("泰坦尼克号：四个关键视角", fontsize=15)
plt.show()
print("  ^ 左下角那条曲线是经典的『家庭规模 U 形』：独自一人和超大家庭都低，中间最高")

### 🔹 小练习 9 · 两分钟

把 `subplots(2, 2, ...)` 改成 `subplots(4, 1, figsize=(10, 12))`，然后把
`axes[0, 0]`、`axes[0, 1]`、`axes[1, 0]`、`axes[1, 1]` 依次改成 `axes[0]`、`axes[1]`、`axes[2]`、`axes[3]`。

**你应该观察到**：四个图变成了竖排，**每个轴标签都没被挤掉**——
`constrained_layout=True` 会自动处理。

> 顺带记住上一课那个坑：**4×1 时 `axes` 是一维数组，写 `axes[0, 1]` 会报错。**

In [ ]:
# 你的代码
fig, axes = plt.subplots(4, 1, figsize=(10, 12), constrained_layout=True)

sex_survival = df.groupby("Sex")["Survived"].mean()
axes[0].bar(["女性", "男性"], sex_survival.values, color=["#DD8452", "#4C72B0"])
axes[0].set_title("性别 vs 生还率")
axes[0].set_ylabel("生还率")
axes[0].set_ylim(0, 1)
for i, value in enumerate(sex_survival.values):
    axes[0].text(i, value + 0.03, f"{value:.0%}", ha="center")

axes[1].hist(df["Fare"], bins=40, color="#55A868", edgecolor="white")
axes[1].set_title("票价分布")
axes[1].set_xlabel("票价")
axes[1].set_ylabel("人数")

df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
family = df.groupby("FamilySize")["Survived"].agg(["count", "mean"])
family = family[family["count"] >= 5]  # 样本太少的组没有统计意义
axes[2].plot(family.index, family["mean"], marker="o", color="#C44E52")
axes[2].set_title("家庭规模 vs 生还率")
axes[2].set_xlabel("家庭人数")
axes[2].set_ylabel("生还率")
axes[2].set_ylim(0, 0.9)
axes[2].grid(alpha=0.3)

port = df.groupby("Embarked")["Survived"].mean().sort_values()
axes[3].barh(port.index, port.values, color="#8172B3")
axes[3].set_title("登船港 vs 生还率")
axes[3].set_xlabel("生还率")
axes[3].set_xlim(0, 0.7)

fig.suptitle("泰坦尼克号：四个关键视角", fontsize=15)
plt.show()
print("  ^ 左下角那条曲线是经典的『家庭规模 U 形』：独自一人和超大家庭都低，中间最高")

## 5. 保存图片

**顺序很重要：先 `savefig` 再 `show`。** 反过来的话，`show()` 会清空画布，存出来是空白图。

In [ ]:
from pathlib import Path

out_dir = Path("../outputs")
out_dir.mkdir(exist_ok=True)

survival = df.groupby("Pclass")["Survived"].mean()
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(["1 等", "2 等", "3 等"], survival.values, color="#4C72B0")
ax.set_title("各舱位生还率")
ax.set_ylabel("生还率")
ax.set_ylim(0, 0.75)

target = out_dir / "survival_by_class.png"
fig.savefig(target, dpi=150, bbox_inches="tight")
print(f"已保存: {target}  ({target.stat().st_size / 1024:.0f} KB)")
plt.show()

### 🔹 小练习 10 · 两分钟

把上面那张图分别以 `dpi=30` 和 `dpi=300` 存成两个文件，**对比文件大小**。

**你应该观察到**：`dpi=30` 只有几 KB 而且糊得看不清字；`dpi=300` 有几百 KB 但非常清晰。

> **实用结论**：`dpi=150` 是写文档、放报告的平衡点；
> 要打印或放大看细节时才用 300。放进 GitHub README 的图，150 就够了。

In [ ]:
# 你的代码from pathlib import Path

out_dir = Path("../outputs")
out_dir.mkdir(exist_ok=True)

survival = df.groupby("Pclass")["Survived"].mean()
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(["1 等", "2 等", "3 等"], survival.values, color="#4C72B0")
ax.set_title("各舱位生还率")
ax.set_ylabel("生还率")
ax.set_ylim(0, 0.75)

target = out_dir / "survival_by_class_300.png"
fig.savefig(target, dpi=300, bbox_inches="tight")
print(f"已保存: {target}  ({target.stat().st_size / 1024:.0f} KB)")
plt.show()

## 6. 图怎么骗人

下面两张图用的是**完全一样的真实数据**，没有伪造任何一个数字。先自己看，
想想你从每张图里分别得出了什么结论。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
labels_ = ["1 等舱", "2 等舱", "3 等舱"]

axes[0].bar(labels_, survival.values, color="#4C72B0")
axes[0].set_title("同一份数据：y 轴从 0 开始", fontsize=12)
axes[0].set_ylabel("生还率")
axes[0].set_ylim(0, 0.75)

axes[1].bar(labels_, survival.values, color="#C44E52")
axes[1].set_title("同一份数据：y 轴从 0.2 开始", fontsize=12)
axes[1].set_ylabel("生还率")
axes[1].set_ylim(0.2, 0.7)  # ← 唯一的差别

fig.tight_layout()
plt.show()
print(f"真实的三个数值: {dict(zip(labels_, survival.round(3).values))}")

左图：三等舱低，但差距是"明显"这个级别。
右图：三等舱几乎贴着底，"1 等舱是三等舱的三倍"——**观感被放大了好几倍**。

真相是 1 等 63%、2 等 47%、3 等 24%。两图都没说谎，
**但右图让人记住的是"三等舱几乎没有生还希望"，这是错的。**

### 常见的四种"诚实撒谎法"

| 手法 | 怎么骗人 | 怎么识别 |
| :--- | :--- | :--- |
| **截断 y 轴** | 不从 0 开始，放大微小差异 | 看纵轴起点是不是 0 |
| **只截取一段数据** | 隐藏前后趋势，制造假"拐点" | 看横轴范围完不完整 |
| **双 y 轴** | 两条无关曲线的交点看起来像"关键转折" | 看两边单位是否真的可比 |
| **换统计口径** | 换个算法，结论就反过来 | 问"换一种算法会不会变" |

### 🔹 小练习 11 · 三分钟 · 自己动手夸张一次

把 `axes[1].set_ylim(0.2, 0.7)` 改成 `axes[1].set_ylim(0.2, 0.3)`，跑一次。

**你应该观察到**：2 等舱和 3 等舱的柱子**顶出了画布**，完全看不见了。
这时候这张图"说"的是：1 等舱 63%，其他两个约等于零。

> **这就是"诚实撒谎"的极限形态**——数据一个字没改，结论完全反了。
> 你以后看到任何一张 y 轴不从 0 开始的图，第一反应应该是：**它为什么这么画？**

In [ ]:
# 你的代码
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
labels_ = ["1 等舱", "2 等舱", "3 等舱"]

axes[0].bar(labels_, survival.values, color="#4C72B0")
axes[0].set_title("同一份数据：y 轴从 0 开始", fontsize=12)
axes[0].set_ylabel("生还率")
axes[0].set_ylim(0, 0.75)

axes[1].bar(labels_, survival.values, color="#C44E52")
axes[1].set_title("同一份数据：y 轴从 0.2 开始", fontsize=12)
axes[1].set_ylabel("生还率")
axes[1].set_ylim(0.2, 0.3)  # ← 唯一的差别

fig.tight_layout()
plt.show()
print(f"真实的三个数值: {dict(zip(labels_, survival.round(3).values))}")

### 作为作者，三条自律

1. **y 轴默认从 0 开始。** 要截断，必须在标题或轴标签里写明原因
   （比如"为看清差异，y 轴从 0.2 开始"）。
2. **不要为了让某个结论成立而挑区间。** 先画全量，再看局部。
3. **双 y 轴要极其谨慎。** 它几乎总在暗示一种数据里不存在的关系。

## 7. Jupyter 工作流：几个省时间的技巧

In [ ]:
# 1) %timeit：测一小段代码跑多久（第 1 课用过它的模块版 timeit）
%timeit df.groupby("Pclass")["Survived"].mean()

In [ ]:
%%time
# 测整个单元格（两个百分号 = 单元格魔法，必须放在第一行开头）
total = 0
for _ in range(3):
    total += df["Fare"].sum()
print("算完了，总计:", round(total, 2))

**另外三件事值得知道**：

1. **导出报告**：`File` → `Save and Export Notebook As` → HTML 或 PDF。
   HTML 最适合分享，别人不用装 Python 就能看，图也在里面。
2. **清空输出**：`Edit` → `Clear Outputs of All Cells`（提交前记得用）。
3. **`%%time` 必须写在单元格第一行**——写在第二行会报 `UsageError`。
   这个小坑我自己踩过，所以专门提一句。

## 8. 综合大练习（约 1 小时）

前面 11 个小练习是"练手"，这 4 道是"交付"——每道都要能拿给别人看。

**统一要求**：每张图都要有标题、轴标签、以及**一条指向重点的注释**。
缺了注释，读者就会自己编一个结论。

### 大练习 1 · 防骗实验（最重要的一题）

自己制造一张"**诚实但误导**"的图：

1. 用**真实数据**（不许改数字）
2. 通过调整 y 轴范围，让**两个差异很小的群体**看起来差异巨大
3. 在同一张画布上并排画出**诚实版**和**误导版**
4. 写下：如果你**只**看到误导版，会得出什么错误结论？

> 建议素材：只有 10–20 人的小群体（比如 60 岁以上的女性）。
> 人数少时比例波动大，截断 y 轴后差异会被放大得特别夸张。
>
> **做完这题你就不会被别人的图骗了——因为你亲手骗过一次人。**

In [ ]:
# 你的代码
labels = ["男性2等舱", "男性3等舱"]
values = [0.157, 0.135]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(labels, values, color="#4C72B0")
axes[0].set_title("诚实版")
axes[0].set_xlabel("群体")
axes[0].set_ylabel("生还率")
axes[0].set_ylim(0, 1)
axes[1].bar(labels, values, color="#C44E52")
axes[1].set_title("误导版")
axes[1].set_xlabel("群体")
axes[1].set_ylabel("生还率")
axes[1].set_ylim(0.13, 0.175)

# 我的错误结论（如果只看到误导版的话）：
# 男性2等仓的生还率明显高于男性3等仓很多

### 大练习 2 · 一张能放进报告的图（串起第 3 课）

做一张**符合项目一要求**的图：

1. 用 `pd.cut` 把年龄分成 5 段（第 4 课学过分箱）
2. 用 `groupby` 算出每段的**人数和生还率**（第 3 课学的）
3. 画成柱状图，**人数作为标注写进柱子里**
4. 加一条注释，指出你发现的最有意思的那一点
5. 保存到 `outputs/`，dpi=150

> 这题是项目一的预演——项目一的"那张能讲故事的图"就是这个要求。

In [ ]:
# 你的代码
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False  # 不设的话负号会变成方框
plt.rcParams["figure.dpi"] = 110

df = pd.read_csv("../data/raw/titanic.csv")

labels = ["幼年", "青年", "中年", "老年", "60岁以上"]
df["年龄段"] = pd.cut(df["Age"], bins=[0, 15, 30, 45, 60, 100], labels=labels)
survival = df.groupby("年龄段")["Survived"].mean()
counts = df.groupby("年龄段")["Survived"].count()

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(labels, survival.values, color="#4C72B0")
ax.set_xlabel("年龄段")
ax.set_ylabel("生还率")
ax.set_ylim(0, 0.6)
for bar, count, value in zip(bars, counts.values, survival.values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.02, f"{count}人", ha="center")
ax.annotate(
    "幼年的生还率比60岁以上的人高了很多",
    xy=(0, 0.5),
    xytext=(1, 0.5),
    arrowprops=dict(arrowstyle="->"),
)

out_dir = pathlib.Path("../outputs")
out_dir.mkdir(exist_ok=True)
target = out_dir / "survival_by_age_group.png"
fig.savefig(target, dpi=150, bbox_inches="tight")
print(f"已保存: {target}  ({target.stat().st_size / 1024:.0f} KB)")

fig.tight_layout()
plt.show()

### 大练习 3 · 2×2 面板图 + 保存

做一张 2×2 的图，四个子图**自选**，但要求：

1. 四个子图用**四种不同**的图形（柱状、直方、散点、箱线、折线里挑四个）
2. 加总标题
3. 保存成 PNG 到 `outputs/`，dpi=150
4. **保存后去文件夹里打开看一眼**，确认不是空白、中文没乱码

> 第 4 步不是多余的——"先 savefig 再 show"这个顺序搞反，存出来就是空白图。
> 我见过不少人提交了空白图还不知道。

In [ ]:
# 你的代码
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

survival = df.groupby("Sex")["Survived"].mean()
bars = axes[0, 0].bar(survival.index, survival.values, color=["#4C72B0", "#C44E52"])
axes[0, 0].set_title("性别vs生存率")
axes[0, 0].set_xlabel("性别")
axes[0, 0].set_ylabel("生存率")
axes[0, 0].set_ylim(0, 1)
for bar, value in zip(bars, survival.values):
    axes[0, 0].text(
        bar.get_x() + bar.get_width() / 2, value + 0.02, f"{value:.0%}", ha="center"
    )

axes[0, 1].hist(df["Fare"], bins=20, color="#4C72B0", edgecolor="white")
axes[0, 1].set_title("票价分布")
axes[0, 1].set_xlabel("票价")
axes[0, 1].set_ylabel("人数")

df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
family = df.groupby("FamilySize")["Survived"].mean()
axes[1, 0].plot(family.index, family.values, marker="o", color="#4C72B0", linestyle="-")
axes[1, 0].set_title("家庭大小vs生存率")
axes[1, 0].set_xlabel("家庭大小")
axes[1, 0].set_ylabel("生存率")
axes[1, 0].grid(alpha=0.3)

port = df.groupby("Embarked")["Survived"].mean()
axes[1, 1].barh(port.index, port.values, color=["#4C72B0", "#C44E52", "#9467BD"])
axes[1, 1].set_title("登船港口vs生存率")
axes[1, 1].set_xlabel("生存率")
axes[1, 1].set_ylabel("登船港口")

fig.suptitle(" Titanic 生还情况分析", fontsize=16)

out_dir = pathlib.Path("../outputs")
out_dir.mkdir(exist_ok=True, parents=True)

target = out_dir / "titanic_analysis.png"
fig.savefig(target, dpi=150, bbox_inches="tight")
print(f"已保存: {target}  ({target.stat().st_size / 1024:.0f} KB)")

fig.tight_layout()
plt.show()

### 大练习 4 · 挑一张图，给它加个"让人一眼看懂"的注释

从前面所有图里挑一张，用 `ax.annotate()` 指向**你最想让读者看到的那一个点**，
并用一句话说明它为什么值得注意。

> 这是"会画图"和"会表达"的分界线。同样一张图，加了这条注释，
> 读者 3 秒抓到重点；没有它，读者要自己找半天，然后得出一个你没想说的结论。
>
> **项目一的那张"能讲故事的图"，评分点就在这一条注释上。**

In [ ]:
# 你的代码
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
family = df.groupby("FamilySize")["Survived"].mean()
fig, axes = plt.subplots(figsize=(8, 6))
axes.plot(family.index, family.values, marker="o", color="#4C72B0", linestyle="-")
axes.set_title("家庭大小vs生存率")
axes.set_xlabel("家庭大小")
axes.set_ylabel("生存率")
axes.grid(alpha=0.3)
axes.annotate(
    "家庭人数为4的生存率比其他家庭人数高很多",
    xy=(4, 0.7),
    xytext=(5, 0.5),
    arrowprops=dict(arrowstyle="->"),
)

plt.show()

## 9. 总结卡片

1. **`fig` 是画布，`ax` 是坐标系。** 所有操作挂在 `ax` 上，别用 `plt.plot()` 的快捷写法。
2. **选图看问题**：随时间→折线；比类别→柱状；看分布→直方图；看关系→散点；比分布→箱线图。
3. **柱状图 vs 直方图**：横轴能不能换顺序。能换是柱状，不能换是直方图。
4. **直方图的 bins 会改变你看到的形状**——别人给你看直方图，先问 bins 是多少。
5. **一张合格的图至少四样**：标题、轴标签、图例、注释。缺了注释，读者自己编结论。
6. **涉及正负的数据必须用发散色系**（RdBu_r / coolwarm），顺序色系会藏掉"正负"这个信息。
7. **y 轴默认从 0 开始。** 截断是放大差异最快的办法，也最容易被识破。
8. **先 `savefig` 再 `show`**，参数 `dpi=150, bbox_inches="tight"`。

---

### 下一课预告

今天你写 20 行代码才画出一张能看的图。下一课会看到：
**`seaborn` 一行就能画出同样质量的图，`plotly` 还能让图动起来。**

但那不是"不用学 matplotlib 了"——**先会手写，才看得懂 seaborn 帮你做了什么**，
也才在它出问题时有能力改。和我们学 numpy→pandas 是同一个道理。

## 10. 卡住时的提示

**小练习 2**：`fig, axes = plt.subplots(1, 2, figsize=(8, 3))`，
然后 `axes[0].plot(x, x**2)`、`axes[1].plot(x, x**3)`，最后只写一次 `plt.show()`。

**小练习 3**：直接复制那一格，改一处跑一次。
重点是**亲眼看第三处改动带来的观感变化**，不用写什么结论。

**小练习 4**：`ax.plot(months, orders, marker="s", color="red", linestyle="--")`。
删 ylim 那一行就是直接注释掉。

**小练习 5**：`fig, axes = plt.subplots(1, 3, figsize=(13, 3))`，
然后 `axes[0].hist(df["Fare"], bins=5)` …… 三个格子写法一样，只换 bins。

**小练习 6**：只改 `alpha` 那一个数字，跑两次。

**小练习 7**：把 `ax.set_ylim(0, 300)` 前面加 `#`。对比之后再删掉 `#`。

**小练习 8**：只改 `cmap` 那一个字符串。

**小练习 9**：改 `subplots` 的参数 + 改四处 `axes[...]` 的写法。
注意 4×1 时是一维数组，只能用 `axes[0]` 这种单下标。

**小练习 10**：`fig.savefig("../outputs/low_dpi.png", dpi=30)` 和 `dpi=300` 两个文件，
在文件夹里对比大小。

**小练习 11**：只改 ylim 的两个数字，跑一次，看柱子是不是顶出去了。

**大练习 1**：① 先画诚实版 `ax.set_ylim(0, 1)`；② 复制一份改成 `(0.3, 0.8)`；
③ 用 `subplots(1, 2)` 并排。选小样本群体（如 60 岁以上女性）效果最明显。

**大练习 2**：`pd.cut(df["Age"], bins=[0,10,20,40,60,100], labels=[...])` →
`groupby` → `ax.bar`。人数写进柱子用 `ax.text`，照抄第 2 节那张图的写法。

**大练习 3**：`subplots(2, 2, figsize=(10, 7), constrained_layout=True)`，
总标题用 `fig.suptitle()`，保存用 `fig.savefig("../outputs/xxx.png", dpi=150, bbox_inches="tight")`。

**大练习 4**：`ax.annotate("文字", xy=(指向的点), xytext=(文字位置), arrowprops=dict(arrowstyle="->"))`。
`xy` 是箭头指的地方，`xytext` 是文字的位置，调几次到好看为止。